In [ ]:
pip install -U transformers torch accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 92.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import json
import os
import time
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
)

# -----------------------------
# Configuración y utilidades
# -----------------------------

@dataclass
class ChatConfig:
    model_name: str = "microsoft/DialoGPT-medium"  # Alternativa: "facebook/blenderbot-400M-distill"
    max_history_turns: int = 8                      # Cuántos turnos guardas (usuario+bot)
    max_new_tokens: int = 120
    temperature: float = 0.8
    top_p: float = 0.92
    repetition_penalty: float = 1.12
    do_sample: bool = True
    device: str = "auto"                            # "auto", "cpu", "cuda"
    persona: str = (
        "Eres un asistente conversacional útil, amable y conciso. "
        "Respondes en español. Si no sabes algo, lo dices con honestidad."
    )

class ChatMemory:
    """
    Memoria simple: lista de turnos [{role: 'user'|'bot', 'text': ...}, ...]
    """
    def __init__(self, max_turns: int = 8):
        self.max_turns = max_turns
        self.turns: List[Dict[str, str]] = []

    def add(self, role: str, text: str):
        self.turns.append({"role": role, "text": text})
        # Conserva solo los últimos N turnos (cada turno = 1 mensaje)
        if len(self.turns) > self.max_turns * 2:
            self.turns = self.turns[-self.max_turns * 2 :]

    def reset(self):
        self.turns = []

    def to_dict(self):
        return {"max_turns": self.max_turns, "turns": self.turns}

    @staticmethod
    def from_dict(d: dict):
        mem = ChatMemory(max_turns=d.get("max_turns", 8))
        mem.turns = d.get("turns", [])
        return mem


class TransformerChatbot:
    def __init__(self, cfg: ChatConfig):
        self.cfg = cfg
        self.device = self._resolve_device(cfg.device)
        self.memory = ChatMemory(max_turns=cfg.max_history_turns)

        self.is_seq2seq = self._is_seq2seq_model(cfg.model_name)

        if self.is_seq2seq:
            self.tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(cfg.model_name).to(self.device)
        else:
            self.tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
            self.model = AutoModelForCausalLM.from_pretrained(cfg.model_name).to(self.device)

        # Algunos tokenizers (DialoGPT) usan eos_token como padding
        if self.tokenizer.pad_token is None and self.tokenizer.eos_token is not None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def _resolve_device(self, device: str) -> torch.device:
        if device == "auto":
            return torch.device("cuda" if torch.cuda.is_available() else "cpu")
        return torch.device(device)

    def _is_seq2seq_model(self, name: str) -> bool:
        # Heurística simple: BlenderBot suele ser seq2seq
        lower = name.lower()
        return "blenderbot" in lower or "t5" in lower or "bart" in lower

    def _build_prompt(self, user_text: str) -> str:
        """
        Construye un “prompt” con persona + historial.
        Para seq2seq usamos un formato de diálogo simple.
        Para causalLM (DialoGPT), concatenamos turnos.
        """
        # Formato legible:
        # Sistema: ...
        # Usuario: ...
        # Bot: ...
        lines = [f"Sistema: {self.cfg.persona}"]

        for t in self.memory.turns:
            if t["role"] == "user":
                lines.append(f"Usuario: {t['text']}")
            else:
                lines.append(f"Bot: {t['text']}")

        lines.append(f"Usuario: {user_text}")
        lines.append("Bot:")
        return "\n".join(lines)

    @torch.no_grad()
    def generate(self, user_text: str) -> str:
        prompt = self._build_prompt(user_text)

        if self.is_seq2seq:
            inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True).to(self.device)
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.cfg.max_new_tokens,
                do_sample=self.cfg.do_sample,
                temperature=self.cfg.temperature,
                top_p=self.cfg.top_p,
                repetition_penalty=self.cfg.repetition_penalty,
            )
            out = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)

            # En seq2seq, el out puede contener parte del prompt dependiendo del modelo.
            # Intentamos extraer lo último “después de Bot:”
            if "Bot:" in out:
                out = out.split("Bot:")[-1].strip()
            return out.strip()

        else:
            # Causal LM: generamos continuación del prompt
            inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, padding=True).to(self.device)

            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.cfg.max_new_tokens,
                do_sample=self.cfg.do_sample,
                temperature=self.cfg.temperature,
                top_p=self.cfg.top_p,
                repetition_penalty=self.cfg.repetition_penalty,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )

            decoded = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)

            # Cortamos para quedarnos sólo con lo que generó tras "Bot:"
            if "Bot:" in decoded:
                answer = decoded.split("Bot:")[-1].strip()
            else:
                # fallback: quita el prompt
                answer = decoded.replace(prompt, "").strip()

            # Limpieza: en algunos modelos puede repetir "Usuario:"; cortamos si aparece
            for stopper in ["Usuario:", "Sistema:"]:
                if stopper in answer:
                    answer = answer.split(stopper)[0].strip()

            return answer.strip()

    def chat_once(self, user_text: str) -> str:
        reply = self.generate(user_text)
        # Actualiza memoria
        self.memory.add("user", user_text)
        self.memory.add("bot", reply)
        return reply

    # -----------------------------
    # Persistencia
    # -----------------------------
    def save(self, path: str):
        data = {
            "config": asdict(self.cfg),
            "memory": self.memory.to_dict()
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

    def load(self, path: str):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        # No cambiamos modelo automáticamente aquí para evitar confusiones;
        # sólo cargamos memoria.
        self.memory = ChatMemory.from_dict(data.get("memory", {}))

def print_help():
    print("""
Comandos:
  /help        Muestra esta ayuda
  /reset       Borra la memoria del chat
  /save ruta   Guarda memoria y config en un JSON
  /load ruta   Carga memoria desde un JSON
  /exit        Salir
""".strip())

def main():
    cfg = ChatConfig(
        model_name=os.environ.get("CHAT_MODEL", "microsoft/DialoGPT-medium"),
        max_history_turns=8,
        max_new_tokens=140,
        temperature=0.85,
        top_p=0.92,
        repetition_penalty=1.12,
        device="auto",
        persona=(
            "Eres un chatbot amigable y útil. "
            "Respondes SIEMPRE en español. "
            "Eres claro, directo y haces preguntas cuando falta información."
        )
    )

    bot = TransformerChatbot(cfg)

    print(f"🤖 Modelo: {cfg.model_name}")
    print(f"🧠 Dispositivo: {bot.device}")
    print("Escribe /help para comandos. Escribe /exit para salir.\n")

    while True:
        user = input("Tú: ").strip()
        if not user:
            continue

        # Comandos
        if user.startswith("/"):
            parts = user.split(maxsplit=1)
            cmd = parts[0].lower()

            if cmd == "/help":
                print_help()
                continue
            if cmd == "/reset":
                bot.memory.reset()
                print("✅ Memoria borrada.")
                continue
            if cmd == "/save":
                path = parts[1] if len(parts) > 1 else "chat_state.json"
                bot.save(path)
                print(f"✅ Guardado en: {path}")
                continue
            if cmd == "/load":
                path = parts[1] if len(parts) > 1 else "chat_state.json"
                bot.load(path)
                print(f"✅ Cargado desde: {path}")
                continue
            if cmd == "/exit":
                print("👋 ¡Hasta luego!")
                break

            print("❓ Comando no reconocido. Escribe /help")
            continue

        start = time.time()
        reply = bot.chat_once(user)
        dt = time.time() - start

        print(f"Bot: {reply}")
        print(f"(t={dt:.2f}s)\n")

if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

... (2 outputs left out)

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

🤖 Modelo: microsoft/DialoGPT-medium
🧠 Dispositivo: cpu
Escribe /help para comandos. Escribe /exit para salir.

Tú: help
Bot: no seu verdad.
(t=102.01s)



KeyboardInterrupt: Interrupted by user

In [1]:
import re
import random
from datetime import datetime

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


class SimpleTransformerChatBot:
    """
    Chatbot estilo 'simple' (como el primero) con intents por reglas.
    Si no matchea ninguna regla -> usa un Transformer (DialoGPT) como fallback.
    """

    def __init__(self, bot_name="SimpleBot", model_name="microsoft/DialoGPT-medium",
                 max_history_tokens=800, max_new_tokens=120):
        self.bot_name = bot_name
        self.user_name = None

        # ===== Reglas (igual que el primero) =====
        self.intents = [
            {
                "tag": "saludo",
                "patterns": [r"\b(hola|hey|buenas|buenos dias|buenas tardes|buenas noches)\b"],
                "responses": [
                    "¡Hola! ¿Cómo estás?",
                    "¡Hey! ¿Qué tal?",
                    "¡Buenas! ¿En qué te ayudo?"
                ]
            },
            {
                "tag": "despedida",
                "patterns": [r"\b(adios|bye|hasta luego|nos vemos|chao)\b"],
                "responses": [
                    "¡Hasta luego! 👋",
                    "¡Nos vemos! Cuídate.",
                    "¡Bye! Fue un gusto."
                ],
                "end_conversation": True
            },
            {
                "tag": "ayuda",
                "patterns": [r"\b(ayuda|que puedes hacer|opciones|comandos)\b"],
                "responses": [
                    "Puedo: saludarte, decir la hora, contar un chiste, recordar tu nombre. "
                    "Si escribes algo fuera de eso, usaré IA para contestar. "
                    "Prueba: 'hora', 'chiste', 'me llamo Ana', 'reset'."
                ]
            },
            {
                "tag": "hora",
                "patterns": [r"\b(hora|que hora es|dime la hora)\b"],
                "responses": [
                    "Son las {time}.",
                    "Ahora mismo: {time}."
                ]
            },
            {
                "tag": "chiste",
                "patterns": [r"\b(chiste|broma|hazme reir)\b"],
                "responses": [
                    "¿Qué hace una abeja en el gimnasio? ¡Zum-ba! 🐝",
                    "¿Por qué el libro se suicidó? Porque tenía demasiadas páginas en blanco 😅",
                    "¿Qué le dijo un bit a otro? Nos vemos en el bus."
                ]
            },
            {
                "tag": "nombre",
                "patterns": [r"\b(me llamo|mi nombre es|soy)\s+([a-zA-ZáéíóúñÁÉÍÓÚÑ]+)\b"],
                "responses": [
                    "¡Encantado, {user}! ¿En qué te ayudo hoy?",
                    "¡Mucho gusto, {user}! ¿Qué quieres hacer?"
                ]
            },
            {
                "tag": "como_estas",
                "patterns": [r"\b(como estas|qué tal|que tal estas|todo bien)\b"],
                "responses": [
                    "¡Bien! Gracias por preguntar 😊 ¿Y tú?",
                    "Todo bien por acá. ¿Cómo va tu día?"
                ]
            },
            {
                "tag": "reset",
                "patterns": [r"\b(reset|reinicia|borra memoria)\b"],
                "responses": [
                    "Listo ✅ reinicié la memoria de conversación.",
                    "Hecho ✅ memoria reiniciada."
                ],
                "reset_conversation": True
            },
        ]

        # ===== Fallback de reglas (si no queremos usar IA) =====
        self.fallback_responses = [
            "No estoy seguro de entender. ¿Puedes reformularlo?",
            "Interesante 🤔. ¿Me lo dices de otra forma?",
            "Prueba 'ayuda' para ver qué entiendo."
        ]

        # ===== Transformer (fallback inteligente) =====
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name).to(self.device)
        self.model.eval()

        # Para DialoGPT, eos sirve como pad
        if self.tokenizer.pad_token is None and self.tokenizer.eos_token is not None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.max_history_tokens = max_history_tokens
        self.max_new_tokens = max_new_tokens

        # Historial en tokens para DialoGPT
        self.chat_history_ids = None

        # “Persona” simple para orientar respuestas (se antepone al input)
        self.persona = (
            "Responde en español. Sé amable, breve y útil. "
            "Si falta información, haz una pregunta."
        )

    # -----------------------------
    # Motor por reglas
    # -----------------------------
    def _match_intent(self, text):
        text = text.lower().strip()
        for intent in self.intents:
            for pattern in intent["patterns"]:
                m = re.search(pattern, text, re.IGNORECASE)
                if m:
                    return intent, m
        return None, None

    # -----------------------------
    # Transformer fallback
    # -----------------------------
    @torch.no_grad()
    def _generate_transformer_reply(self, user_text: str) -> str:
        """
        Genera respuesta con DialoGPT usando historial (chat_history_ids).
        """
        # Le metemos “persona” al mensaje del usuario para empujarlo al español
        # (DialoGPT es muy flexible pero esto ayuda).
        prompt = f"{self.persona}\nUsuario: {user_text}\nBot:"

        new_user_input_ids = self.tokenizer.encode(
            prompt + self.tokenizer.eos_token, return_tensors="pt"
        ).to(self.device)

        # Concatena historial si existe
        if self.chat_history_ids is not None:
            bot_input_ids = torch.cat([self.chat_history_ids, new_user_input_ids], dim=-1)
        else:
            bot_input_ids = new_user_input_ids

        # Recorta si el historial se volvió enorme
        if bot_input_ids.shape[-1] > self.max_history_tokens:
            bot_input_ids = bot_input_ids[:, -self.max_history_tokens :]

        output_ids = self.model.generate(
            bot_input_ids,
            max_new_tokens=self.max_new_tokens,
            do_sample=True,
            temperature=0.85,
            top_p=0.92,
            repetition_penalty=1.12,
            pad_token_id=self.tokenizer.eos_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )

        # Actualiza historial
        self.chat_history_ids = output_ids

        # Decodifica SOLO lo nuevo (lo generado al final)
        generated = output_ids[:, bot_input_ids.shape[-1]:]
        reply = self.tokenizer.decode(generated[0], skip_special_tokens=True).strip()

        # Limpieza básica
        reply = reply.replace("Usuario:", "").replace("Bot:", "").strip()
        if not reply:
            reply = random.choice(self.fallback_responses)
        return reply

    # -----------------------------
    # Respuesta principal
    # -----------------------------
    def respond(self, text: str):
        intent, match = self._match_intent(text)

        if not intent:
            # Si no matchea, usamos Transformer
            reply = self._generate_transformer_reply(text)

            # Personalización por nombre (si existe)
            if self.user_name and random.random() < 0.25:
                reply = f"{self.user_name}, {reply}"
            return reply, False

        # Reset de conversación (incluye historial del transformer)
        if intent.get("reset_conversation"):
            self.chat_history_ids = None
            return random.choice(intent["responses"]), False

        # Manejo nombre
        if intent["tag"] == "nombre":
            self.user_name = match.group(2)
            return random.choice(intent["responses"]).format(user=self.user_name), False

        # Hora
        if intent["tag"] == "hora":
            now = datetime.now().strftime("%H:%M:%S")
            return random.choice(intent["responses"]).format(time=now), False

        # Respuesta por regla
        reply = random.choice(intent["responses"])
        end = intent.get("end_conversation", False)
        return reply, end


def main():
    bot = SimpleTransformerChatBot(
        bot_name="SimpleBot",
        model_name="microsoft/DialoGPT-medium",  # puedes cambiar a: "microsoft/DialoGPT-small"
        max_history_tokens=800,
        max_new_tokens=120
    )

    print(f"{bot.bot_name}: ¡Hola! (Escribe 'ayuda' para opciones, 'reset' para reiniciar, 'adios' para salir)")
    print(f"(Dispositivo: {bot.device})\n")

    while True:
        user = input("Tú: ").strip()
        if not user:
            continue

        reply, end = bot.respond(user)
        print(f"{bot.bot_name}: {reply}\n")

        if end:
            break


if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

... (4 outputs left out)

SimpleBot: ¡Hola! (Escribe 'ayuda' para opciones, 'reset' para reiniciar, 'adios' para salir)
(Dispositivo: cpu)

Tú: hola
SimpleBot: ¡Hey! ¿Qué tal?

Tú: ayuda
SimpleBot: Puedo: saludarte, decir la hora, contar un chiste, recordar tu nombre. Si escribes algo fuera de eso, usaré IA para contestar. Prueba: 'hora', 'chiste', 'me llamo Ana', 'reset'.

Tú: dime algo gracioso


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


SimpleBot: Vuelvo, por lo menos, no hay que ver alguien que era mal de eso?



KeyboardInterrupt: Interrupted by user

In [ ]:
import re
import random
from datetime import datetime
from typing import Dict, List, Tuple

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, util


class VectorRouter:
    """
    Router semántico:
    - Precalcula embeddings para ejemplos de cada intent.
    - Para un texto nuevo, calcula similitud coseno y decide el intent.
    """

    def __init__(self, embed_model_name: str, intents: List[Dict], threshold: float = 0.55):
        self.embedder = SentenceTransformer(embed_model_name)
        self.intents = intents
        self.threshold = threshold

        # Aplanamos ejemplos -> [(tag, example_text), ...]
        self.examples: List[Tuple[str, str]] = []
        for it in intents:
            tag = it["tag"]
            for ex in it.get("examples", []):
                self.examples.append((tag, ex))

        # Calculamos embeddings una sola vez
        self.example_texts = [t for _, t in self.examples]
        self.example_tags = [tag for tag, _ in self.examples]
        self.example_emb = self.embedder.encode(self.example_texts, convert_to_tensor=True, normalize_embeddings=True)

    def route(self, text: str) -> Tuple[str, float]:
        """
        Retorna (tag, score). Si score < threshold, tag="__fallback__"
        """
        emb = self.embedder.encode([text], convert_to_tensor=True, normalize_embeddings=True)
        scores = util.cos_sim(emb, self.example_emb)[0]  # vector de similitudes
        best_idx = int(torch.argmax(scores).item())
        best_score = float(scores[best_idx].item())
        best_tag = self.example_tags[best_idx]

        if best_score < self.threshold:
            return "__fallback__", best_score
        return best_tag, best_score


class SimpleVectorizedTransformerChatBot:
    """
    - Intents por embeddings (vectorización)
    - Reglas especiales (nombre/reset) por regex (porque son acciones)
    - Fallback generativo con DialoGPT cuando no hay intent claro
    """

    def __init__(
        self,
        bot_name="SimpleBot",
        gen_model_name="microsoft/DialoGPT-medium",
        embed_model_name="sentence-transformers/all-MiniLM-L6-v2",
        intent_threshold=0.55,
        max_history_tokens=800,
        max_new_tokens=120
    ):
        self.bot_name = bot_name
        self.user_name = None

        # ========= Intents: respuestas + ejemplos (para vectorizar) =========
        self.intents: List[Dict] = [
            {
                "tag": "saludo",
                "examples": ["hola", "buenas", "hey", "buenos días", "buenas tardes", "qué onda"],
                "responses": [
                    "¡Hola! ¿Cómo estás?",
                    "¡Hey! ¿Qué tal?",
                    "¡Buenas! ¿En qué te ayudo?"
                ]
            },
            {
                "tag": "despedida",
                "examples": ["adiós", "bye", "hasta luego", "nos vemos", "me voy", "chao"],
                "responses": [
                    "¡Hasta luego! 👋",
                    "¡Nos vemos! Cuídate.",
                    "¡Bye! Fue un gusto."
                ],
                "end_conversation": True
            },
            {
                "tag": "ayuda",
                "examples": ["ayuda", "qué puedes hacer", "opciones", "comandos", "cómo funcionas"],
                "responses": [
                    "Puedo: saludarte, decir la hora, contar un chiste, recordar tu nombre. "
                    "Si escribes algo fuera de eso, usaré IA para contestar. "
                    "Prueba: 'hora', 'chiste', 'me llamo Ana', 'reset'."
                ]
            },
            {
                "tag": "hora",
                "examples": ["qué hora es", "dime la hora", "hora", "tienes la hora"],
                "responses": [
                    "Son las {time}.",
                    "Ahora mismo: {time}."
                ]
            },
            {
                "tag": "chiste",
                "examples": ["cuéntame un chiste", "broma", "hazme reír", "dime algo gracioso"],
                "responses": [
                    "¿Qué hace una abeja en el gimnasio? ¡Zum-ba! 🐝",
                    "¿Por qué el libro se suicidó? Porque tenía demasiadas páginas en blanco 😅",
                    "¿Qué le dijo un bit a otro? Nos vemos en el bus."
                ]
            },
            {
                "tag": "como_estas",
                "examples": ["cómo estás", "qué tal estás", "todo bien", "cómo te va"],
                "responses": [
                    "¡Bien! Gracias por preguntar 😊 ¿Y tú?",
                    "Todo bien por acá. ¿Cómo va tu día?"
                ]
            },
        ]

        # ========= Router vectorial =========
        self.router = VectorRouter(embed_model_name, self.intents, threshold=intent_threshold)

        # ========= Acciones por regex (porque capturan datos) =========
        self.re_nombre = re.compile(r"\b(me llamo|mi nombre es|soy)\s+([a-zA-ZáéíóúñÁÉÍÓÚÑ]+)\b", re.I)
        self.re_reset = re.compile(r"\b(reset|reinicia|borra memoria)\b", re.I)

        # ========= Transformer generativo (fallback) =========
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
        self.model = AutoModelForCausalLM.from_pretrained(gen_model_name).to(self.device)
        self.model.eval()

        if self.tokenizer.pad_token is None and self.tokenizer.eos_token is not None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.max_history_tokens = max_history_tokens
        self.max_new_tokens = max_new_tokens
        self.chat_history_ids = None

        self.persona = (
            "Responde en español. Sé amable, breve y útil. "
            "Si falta información, haz una pregunta."
        )

        self.fallback_responses = [
            "No estoy seguro de entender. ¿Puedes reformularlo?",
            "Interesante 🤔. ¿Me lo dices de otra forma?",
            "Prueba 'ayuda' para ver qué entiendo."
        ]

    # -----------------------------
    # Fallback generativo con DialoGPT
    # -----------------------------
    @torch.no_grad()
    def _generate_transformer_reply(self, user_text: str) -> str:
        prompt = f"{self.persona}\nUsuario: {user_text}\nBot:"
        new_user_input_ids = self.tokenizer.encode(
            prompt + self.tokenizer.eos_token, return_tensors="pt"
        ).to(self.device)

        if self.chat_history_ids is not None:
            bot_input_ids = torch.cat([self.chat_history_ids, new_user_input_ids], dim=-1)
        else:
            bot_input_ids = new_user_input_ids

        if bot_input_ids.shape[-1] > self.max_history_tokens:
            bot_input_ids = bot_input_ids[:, -self.max_history_tokens:]

        output_ids = self.model.generate(
            bot_input_ids,
            max_new_tokens=self.max_new_tokens,
            do_sample=True,
            temperature=0.85,
            top_p=0.92,
            repetition_penalty=1.12,
            pad_token_id=self.tokenizer.eos_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )

        self.chat_history_ids = output_ids
        generated = output_ids[:, bot_input_ids.shape[-1]:]
        reply = self.tokenizer.decode(generated[0], skip_special_tokens=True).strip()

        reply = reply.replace("Usuario:", "").replace("Bot:", "").strip()
        if not reply:
            reply = random.choice(self.fallback_responses)
        return reply

    # -----------------------------
    # Respuesta principal
    # -----------------------------
    def respond(self, text: str) -> Tuple[str, bool]:
        text = text.strip()
        if not text:
            return "Escribe algo 🙂", False

        # 1) Acciones por regex (más confiables)
        m = self.re_reset.search(text)
        if m:
            self.chat_history_ids = None
            return "Listo ✅ reinicié la memoria de conversación.", False

        m = self.re_nombre.search(text)
        if m:
            self.user_name = m.group(2)
            return f"¡Encantado, {self.user_name}! ¿En qué te ayudo hoy?", False

        # 2) Router vectorial para intents
        tag, score = self.router.route(text)

        # 3) Si no hay intent claro -> Transformer
        if tag == "__fallback__":
            reply = self._generate_transformer_reply(text)
            if self.user_name and random.random() < 0.25:
                reply = f"{self.user_name}, {reply}"
            return reply, False

        # 4) Respuesta por intent
        intent = next(i for i in self.intents if i["tag"] == tag)

        if tag == "hora":
            now = datetime.now().strftime("%H:%M:%S")
            reply = random.choice(intent["responses"]).format(time=now)
        else:
            reply = random.choice(intent["responses"])

        end = intent.get("end_conversation", False)
        return reply, end


def main():
    bot = SimpleVectorizedTransformerChatBot(
        bot_name="SimpleBot",
        gen_model_name="microsoft/DialoGPT-medium",
        embed_model_name="sentence-transformers/all-MiniLM-L6-v2",
        intent_threshold=0.55,       # súbelo a 0.65 si se confunde mucho
        max_history_tokens=800,
        max_new_tokens=120
    )

    print(f"{bot.bot_name}: ¡Hola! (Escribe 'ayuda', 'reset' o 'adios' para salir)")
    print(f"(Dispositivo: {bot.device})\n")

    while True:
        user = input("Tú: ").strip()
        reply, end = bot.respond(user)
        print(f"{bot.bot_name}: {reply}\n")
        if end:
            break


if __name__ == "__main__":
    main()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (6 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SimpleBot: ¡Hola! (Escribe 'ayuda', 'reset' o 'adios' para salir)
(Dispositivo: cpu)

SimpleBot: Listo ✅ reinicié la memoria de conversación.

SimpleBot: Puedo: saludarte, decir la hora, contar un chiste, recordar tu nombre. Si escribes algo fuera de eso, usaré IA para contestar. Prueba: 'hora', 'chiste', 'me llamo Ana', 'reset'.

SimpleBot: Ahora mismo: 00:32:36.

SimpleBot: ¡Encantado, fili! ¿En qué te ayudo hoy?

SimpleBot: Todo bien por acá. ¿Cómo va tu día?

SimpleBot: ¿Por qué el libro se suicidó? Porque tenía demasiadas páginas en blanco 😅

SimpleBot: Escribe algo 🙂

